<a href="https://colab.research.google.com/github/anamacao/FAPESP-PIBIC-scrapping/blob/main/Congreso_de_la_Naci%C3%B3n_Argentina.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!apt-get update
# Remove previous apt-get installations of chromium to avoid conflicts
!apt-get purge chromium-browser chromium-chromedriver -y
!apt-get autoremove -y

# Download and install specific compatible versions of Chrome and ChromeDriver
# These versions are known to work well together in Colab as of a recent check.
# You might need to update these versions if they become outdated.
CHROME_VERSION = '126.0.6478.61' # Example: Use a stable Chrome version
CHROMEDRIVER_VERSION = '126.0.6478.61' # Must match Chrome version

# Download Chrome browser
!wget -N https://edgedl.me.gvt1.com/edgedl/chrome/chrome-for-testing/{CHROME_VERSION}/linux64/chrome-linux64.zip
!unzip -o chrome-linux64.zip -d /opt/
!mv /opt/chrome-linux64 /opt/chrome

# Download ChromeDriver
!wget -N https://edgedl.me.gvt1.com/edgedl/chrome/chrome-for-testing/{CHROMEDRIVER_VERSION}/linux64/chromedriver-linux64.zip
!unzip -o chromedriver-linux64.zip -d /opt/
!mv /opt/chromedriver-linux64 /opt/chromedriver

# Set environment variables to point to the new installations
import os
os.environ['CHROME_PATH'] = '/opt/chrome/chrome'
os.environ['CHROMEDRIVER_PATH'] = '/opt/chromedriver/chromedriver'
os.environ['PATH'] += f":{os.environ['CHROME_PATH']}:{os.environ['CHROMEDRIVER_PATH']}"

# Make chromedriver executable
!chmod +x /opt/chromedriver/chromedriver

print(f"Chrome installed at: {os.environ['CHROME_PATH']}")
print(f"ChromeDriver installed at: {os.environ['CHROMEDRIVER_PATH']}")

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 1s (3,009 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Package 'chromium-browser' is not installed, so no

In [2]:
pip install beautifulsoup4 requests selenium webdriver_manager

In [16]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
import time
import os

url = 'https://icn.gob.ar/noticias'

# Setup Selenium WebDriver
chrome_options = Options()
chrome_options.add_argument('--headless') # Run in headless mode (without opening a browser window)
chrome_options.add_argument('--no-sandbox') # Required for Colab environment
chrome_options.add_argument('--disable-dev-shm-usage') # Required for Colab environment
chrome_options.add_argument('--remote-debugging-port=9222') # Add remote debugging port to avoid crashes
chrome_options.add_argument('--disable-gpu') # Disable GPU hardware acceleration
chrome_options.add_argument('--window-size=1920,1080') # Set a default window size for headless
chrome_options.add_argument('--no-default-browser-check') # Disable default browser check
chrome_options.add_argument('--log-level=3') # Suppress excessive logging
chrome_options.add_argument('--disable-notifications')
chrome_options.add_argument('--disable-extensions')
chrome_options.add_argument('--disable-browser-side-navigation')
chrome_options.add_argument('--disable-features=VizDisplayCompositor') # Often helps in headless environments

# Point to the manually installed chromium binary using the environment variable
chrome_options.binary_location = os.environ['CHROME_PATH']

print(f"Chrome binary location: {os.environ['CHROME_PATH']}")
print(f"ChromeDriver binary location: {os.environ['CHROMEDRIVER_PATH']}")

# Verify executability and existence
if not os.path.exists(os.environ['CHROME_PATH']):
    print(f"Error: Chrome binary not found at {os.environ['CHROME_PATH']}")
if not os.path.exists(os.environ['CHROMEDRIVER_PATH']):
    print(f"Error: ChromeDriver binary not found at {os.environ['CHROMEDRIVER_PATH']}")

# Add service_args for verbose logging from ChromeDriver
service = Service(executable_path=os.environ['CHROMEDRIVER_PATH'],
                  service_args=['--verbose', '--log-path=/tmp/chromedriver.log'])

driver = webdriver.Chrome(service=service, options=chrome_options)

print(f"Navigating to {url} with Selenium...")
driver.get(url)

# Give the page some time to load dynamic content (adjust if needed)
time.sleep(10) # Increased sleep time

# Get the page source after dynamic content has loaded
page_source = driver.page_source
driver.quit() # Close the browser

soup = BeautifulSoup(page_source, 'html.parser')

# Define the CSS selectors provided by the user for news elements
# The JS snippet uses: '.noticia, .post, .entry, .card, [id*="noticia"]'
news_element_selectors = '.noticia, .post, .entry, .card, [id*="noticia"]'

# Find all elements matching these selectors
potential_news_containers = soup.select(news_element_selectors)

article_links = []
base_url = 'https://icn.gob.ar' # Base URL to construct absolute links

for container in potential_news_containers:
    # Find all links within each potential news container
    links_in_container = container.find_all('a', href=True)
    for link in links_in_container:
        href = link['href']
        # Ensure it's a valid link and convert to absolute URL
        if href and not href.startswith('#'): # Avoid anchor links
            if href.startswith('/'):
                full_link = base_url + href
            elif href.startswith(base_url):
                full_link = href
            else:
                continue # Skip external or non-relative/non-absolute links

            article_links.append(full_link)

# Remove duplicates and ensure only HTTP/HTTPS links are processed
unique_article_links = list(set([link for link in article_links if link.startswith('http')]))

print(f"Found {len(unique_article_links)} potential unique article links.")
print("Sample links:")
for i, link in enumerate(unique_article_links[:5]):
    print(f"- {link}")

# news_data will be populated in a subsequent step by visiting these links
# Initialize an empty DataFrame for now
df_news = pd.DataFrame(columns=['title', 'date', 'content_summary', 'link'])
display(df_news.head())

Chrome binary location: /opt/chrome/chrome
ChromeDriver binary location: /opt/chromedriver/chromedriver
Navigating to https://icn.gob.ar/noticias with Selenium...
Found 0 potential unique article links.
Sample links:


,title,date,content_summary,link


In [15]:
import os
from bs4 import BeautifulSoup

# Ensure page_source is available (it should be from previous Selenium run)
if 'page_source' not in locals():
    print("Error: 'page_source' variable not found. Please run the Selenium cell (f10cdb0b) first.")
else:
    soup = BeautifulSoup(page_source, 'html.parser')

    main_content = soup.find(id='mainContent')

    if main_content:
        print("Found #mainContent. Inspecting its structure:\n")

        # Inspect children of mainContent
        print("--- Children of #mainContent (first 5 elements) ---")
        for i, child in enumerate(main_content.find_all(recursive=False)):
            if i >= 5: # Limit output to first 5 children for brevity
                break
            print(f"  Tag: {child.name}, Class: {child.get('class', 'N/A')}, ID: {child.get('id', 'N/A')}")
            print(f"  HTML Snippet: {str(child)[:200]}...") # First 200 chars
            print(f"  Child Tags: {[c.name for c in child.find_all(recursive=False)]}\n")

        # Look for 'a' tags inside mainContent and their parents
        print("--- Links within #mainContent (first 10 links) ---")
        links_found = 0
        for a_tag in main_content.find_all('a', href=True):
            if links_found >= 10: # Limit output to first 10 links for brevity
                break
            parent = a_tag.find_parent()
            grand_parent = parent.find_parent() if parent else None

            print(f"  Link Text: {a_tag.get_text(strip=True)[:50]}...")
            print(f"  Link Href: {a_tag['href']}")
            print(f"  Parent: Tag={parent.name if parent else 'N/A'}, Class={parent.get('class', 'N/A')}, ID={parent.get('id', 'N/A')}")
            print(f"  Grandparent: Tag={grand_parent.name if grand_parent else 'N/A'}, Class={grand_parent.get('class', 'N/A')}, ID={grand_parent.get('id', 'N/A')}\n")
            links_found += 1

        if links_found == 0:
            print("No links found within #mainContent.")

    else:
        print("Error: #mainContent not found in the page source.")


Found #mainContent. Inspecting its structure:

--- Children of #mainContent (first 5 elements) ---
  Tag: div, Class: ['container', 'page1', 'no-backgrond'], ID: pageNoticias
  HTML Snippet: <div class="container page1 no-backgrond" id="pageNoticias">
<div class="heading-block center nobottommargin heading-block-no-border">
<h2 class="titulo">Noticias</h2>
</div>
<div class="row" style="m...
  Child Tags: ['div', 'div']

  Tag: script, Class: N/A, ID: N/A
  HTML Snippet: <script>
$(document).ready(function(){
   setUpNoticias();
   $('#pageNoticias .panel').click(function(e){
	   var data= $(e.target).closest('.panel').data();
	   if(typeof  data.slug=='undefined')
	 ...
  Child Tags: []

--- Links within #mainContent (first 10 links) ---
No links found within #mainContent.


In [13]:
import os
from bs4 import BeautifulSoup

# Assuming 'page_source' is available from the previous Selenium run

# Save the full page source to an HTML file for inspection
html_file_path = 'page_source.html'
with open(html_file_path, 'w', encoding='utf-8') as f:
    f.write(page_source)

print(f"Full page source saved to {html_file_path}")
print("You can download this file from the file browser (left pane) and inspect it.")

soup = BeautifulSoup(page_source, 'html.parser')

# Re-find the articles using the 'stretched' class
articles = soup.find_all(class_='stretched')

print(f"Found {len(articles)} elements with class 'stretched'.")

if articles:
    print("\n--- Inspecting the first 'stretched' element ---")
    first_article = articles[0]
    print(first_article.prettify())
    print("\n--- End of first 'stretched' element ---")
else:
    print("No elements with class 'stretched' found in the page source.")


Full page source saved to page_source.html
You can download this file from the file browser (left pane) and inspect it.
Found 1 elements with class 'stretched'.

--- Inspecting the first 'stretched' element ---
<body class="stretched" style="">
 <div class="clearfix" id="wrapper">
  <div id="header">
   <div class="clearfix" style="max-width:1366px;margin-left:auto;margin-right:auto">
    <div id="logo">
     <a class="standard-logo" href="https://icn.gob.ar/">
      <img alt="Logo Imprenta del Congreso de La Nación" src="https://icn.gob.ar/public/icn/estatico/imagenes/logo.png" tabindex="1"/>
     </a>
     <a class="retina-logo" href="https://icn.gob.ar/">
      <img alt="Logo Imprenta del Congreso de La Nación" src="https://icn.gob.ar/public/icn/estatico/imagenes/logo.png" tabindex="1"/>
     </a>
    </div>
    <div class="rs-menu">
     <i class="fa fa-align-justify">
     </i>
    </div>
    <div id="primary-menu">
     <ul class="sf-js-enabled sf-arrows" id="main-menu">
      <l

In [11]:
import os
from bs4 import BeautifulSoup

# Assuming 'page_source' is already available from the previous run
# If not, you might need to re-run the Selenium cell.

# Save the full page source to an HTML file for inspection
html_file_path = 'page_source.html'
with open(html_file_path, 'w', encoding='utf-8') as f:
    f.write(page_source)

print(f"Full page source saved to {html_file_path}")
print("You can download this file from the file browser (left pane) and inspect it.")

soup = BeautifulSoup(page_source, 'html.parser')

# Attempt to find common elements that might represent articles
# This is a generic search since 'article.et_pb_post' did not yield results.

# Look for div elements that contain an h2 (potential title) and a p (potential date/summary)
potential_articles = []
for div in soup.find_all('div'):
    h2_tag = div.find('h2')
    p_tag = div.find('p')
    if h2_tag and p_tag:
        potential_articles.append(div)
        # Optionally, print a snippet of the first few to avoid excessive output
        if len(potential_articles) < 5:
            print(f"\n--- Potential Article Snippet ---\n{div.prettify()[:500]}...\n")

print(f"Found {len(potential_articles)} div elements containing both h2 and p tags.")

if not potential_articles:
    print("No generic div-h2-p patterns found. Trying more general search...")
    # If the above doesn't work, try looking for any `article` tags without specific classes
    # or `div` tags with common class names for content like 'post', 'item', 'entry'
    generic_articles = soup.find_all(['article', 'div'], class_=lambda x: x and ('post' in x or 'item' in x or 'entry' in x or 'news' in x))
    print(f"Found {len(generic_articles)} generic article/div elements.")
    if generic_articles:
        print("Sample of generic article/div elements:")
        for i, article in enumerate(generic_articles[:3]):
            print(f"\n--- Generic Article/Div Sample {i+1} ---\n{article.prettify()[:500]}...\n")
    else:
        print("No generic article/div elements found either. The content might be in an iframe or loaded differently.")


Full page source saved to page_source.html
You can download this file from the file browser (left pane) and inspect it.
Found 0 div elements containing both h2 and p tags.
No generic div-h2-p patterns found. Trying more general search...
Found 0 generic article/div elements.
No generic article/div elements found either. The content might be in an iframe or loaded differently.


In [7]:
!apt-get update
!apt-get install -y libatk1.0-0 libgtk-3-0 libgdk-pixbuf2.0-0 libxss1 libnss3 libasound2

# Re-run the Selenium setup and scraping cell (f10cdb0b) after installing dependencies

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 1s (3,340 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libxss1 is already the newest version (1:1.2.3-1bu

In [6]:
import os

log_file_path = '/tmp/chromedriver.log'

if os.path.exists(log_file_path):
    print(f"Content of {log_file_path}:")
    with open(log_file_path, 'r') as f:
        print(f.read())
else:
    print(f"ChromeDriver log file not found at {log_file_path}")

Content of /tmp/chromedriver.log:
[1775730243.653][INFO]: Starting ChromeDriver 126.0.6478.61 (8dc092df54ce9b93406cb7fec530eb297bc0b332-refs/branch-heads/6478_56@{#3}) on port 40115
[1775730243.653][INFO]: Please see https://chromedriver.chromium.org/security-considerations for suggestions on keeping ChromeDriver safe.
[1775730243.665][INFO]: [1930c0524c0ec9009bfeb3b5564f30dd] COMMAND InitSession {
   "capabilities": {
      "alwaysMatch": {
         "browserName": "chrome",
         "goog:chromeOptions": {
            "args": [ "--headless", "--no-sandbox", "--disable-dev-shm-usage", "--remote-debugging-port=9222", "--disable-gpu", "--window-size=1920,1080", "--no-default-browser-check", "--log-level=3", "--disable-notifications", "--disable-extensions", "--disable-browser-side-navigation", "--disable-features=VizDisplayCompositor" ],
            "binary": "/opt/chrome/chrome",
            "extensions": [  ]
         },
         "pageLoadStrategy": "normal"
      },
      "firstMatch"